# GATE-CS Doubt Solver: QLoRA Fine-Tuning (Phase 2)

This notebook runs **4-bit QLoRA fine-tuning** of `Qwen/Qwen2.5-1.5B-Instruct` on the curated GATE CS dataset.

### Hardware Specs & Budget
- **Target Hardware**: Free-Tier Google Colab (T4 16GB) or Kaggle (T4 x 2 / P100 16GB)
- **VRAM Footprint**: ~5.8 GB (Base 4-bit weights ~1.1GB, activations ~3.2GB, optimizer states ~1.5GB)
- **Estimated Duration**: ~25–35 minutes for 3 epochs with gradient accumulation

In [ ]:
# Step 1: Install Dependencies
!pip install -q -U torch transformers peft trl bitsandbytes accelerate datasets wandb

In [ ]:
# Step 2: Verify GPU Environment & VRAM
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"bfloat16 Supported: {torch.cuda.is_bf16_supported()}")

In [ ]:
# Step 3: Setup Weights & Biases Logging (Optional / Recommended)
import os
import wandb

# Set WANDB_API_KEY if available or login interactively
wandb.init(project="gate-cs-doubt-solver", name="qwen2.5-1.5b-qlora-sft-v1", reinit=True)

In [ ]:
# Step 4: Load Dataset Splits (Auto-clone repository if on Colab)
import os, json
from datasets import Dataset, DatasetDict

if not os.path.exists("data/splits/train.jsonl"):
    print("Downloading CALYPSO dataset from GitHub...")
    !git clone https://github.com/piyush23-eng/CALYPSO.git temp_repo && cp -r temp_repo/data . && rm -rf temp_repo

def load_split(filepath):
    records = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line.strip()))
    return [{"messages": r["messages"]} for r in records]

train_data = load_split("data/splits/train.jsonl")
val_data = load_split("data/splits/val.jsonl")

dataset = DatasetDict({
    "train": Dataset.from_list(train_data),
    "validation": Dataset.from_list(val_data)
})

print(f"Loaded Train: {len(dataset['train'])}, Validation: {len(dataset['validation'])}")

In [ ]:
# Step 5: Configure 4-bit Quantization & LoRA Adapter
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# 4-Bit NormalFloat Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

# LoRA Adapter Config (All Linear Projections)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

# Load Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True, padding_side="right")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Step 6: Initialize SFTTrainer & Launch Training
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="models/checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,  # Effective Batch Size = 16
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_checkpointing=True,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    seed=42,
    report_to="wandb" if os.environ.get("WANDB_API_KEY") else "none",
    max_seq_length=1024,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    peft_config=peft_config,
)

# Start Training
train_result = trainer.train()
print("Training Complete!")
print(train_result.metrics)

In [ ]:
# Step 7: Save Best LoRA Adapter & Tokenizer
ADAPTER_DIR = "models/gate_qwen_1.5b_lora"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Saved best checkpoint adapter to: {ADAPTER_DIR}")

In [ ]:
# Step 8: Merge LoRA Adapter into Base Model (for Phase 3 Eval & Phase 4 GGUF)
from peft import PeftModel

MERGED_DIR = "models/gate_qwen_1.5b_merged"

print("Reloading base model in float16 for merging...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

peft_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
merged_model = peft_model.merge_and_unload()

merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged standalone model saved to {MERGED_DIR}")